# #30DayMapChallenge
## Day 30: Makeover

### Trial 1

In [ ]:
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
import numpy as np
from pathlib import Path
import re

In [ ]:
# Load all GeoJSON files from the Buildings folder
buildings_folder = Path('Buildings')
building_files = sorted(buildings_folder.glob('NewCairo_Building_Height_*.geojson'))

print(f"Found {len(building_files)} files")

In [ ]:
# Load all years of data
all_buildings = []
for file in building_files:
    # Extract year from filename
    year_match = re.search(r'(\d{4})', file.name)
    if year_match:
        year = int(year_match.group(1))
        
        gdf = gpd.read_file(file)
        gdf['year'] = year
        all_buildings.append(gdf)
        print(f"Loaded {len(gdf)} buildings from {year}")

In [ ]:
# Load all years of data
all_buildings = []
for file in building_files:
    # Extract year from filename
    year_match = re.search(r'(\d{4})', file.name)
    if year_match:
        year = int(year_match.group(1))
        
        gdf = gpd.read_file(file)
        gdf['year'] = year
        all_buildings.append(gdf)
        print(f"Loaded {len(gdf)} buildings from {year}")

In [ ]:
# Combine all data
buildings_all = pd.concat(all_buildings, ignore_index=True)

# Project to a local coordinate system for better visualization
buildings_projected = buildings_all.to_crs('EPSG:3857')  # Web Mercator

# Get the center for visualization (use projected data to avoid warning)
center_point = buildings_projected.geometry.centroid
center_lon = buildings_all.geometry.centroid.x.mean()
center_lat = buildings_all.geometry.centroid.y.mean()

print(f"\nTotal buildings across all years: {len(buildings_all)}")
print(f"Center coordinates: {center_lat:.4f}, {center_lon:.4f}")
print(f"Years: {sorted(buildings_all['year'].unique())}")

In [ ]:
# Create animation frames
years = sorted(buildings_all['year'].unique())
frames = []

for year in years:
    buildings_year = buildings_projected[buildings_projected['year'] == year]
    
    # Create traces for this year
    traces = []
    
    for idx, building in buildings_year.iterrows():
        if building.geometry.geom_type == 'Polygon':
            coords = list(building.geometry.exterior.coords)
            x = [c[0] for c in coords]
            y = [c[1] for c in coords]
            height = building.get('Heightmean', 0)
            
            if height > 0:  # Only show buildings with height
                # Create building as a Mesh3d
                n = len(x)
                
                # Bottom vertices
                x_all = x + x
                y_all = y + y
                z_all = [0] * n + [height] * n
                
                # Create faces (triangles)
                # Bottom face
                i_bottom = [0] * (n - 2)
                j_bottom = list(range(1, n - 1))
                k_bottom = list(range(2, n))
                
                # Top face
                i_top = [n] * (n - 2)
                j_top = [n + i for i in range(2, n)]
                k_top = [n + i for i in range(1, n - 1)]
                
                # Side faces
                i_side = []
                j_side = []
                k_side = []
                for idx in range(n - 1):
                    # Two triangles per side
                    i_side.extend([idx, idx])
                    j_side.extend([idx + 1, n + idx + 1])
                    k_side.extend([n + idx, idx + 1])
                
                i_all = i_bottom + i_top + i_side
                j_all = j_bottom + j_top + j_side
                k_all = k_bottom + k_top + k_side
                
                traces.append(go.Mesh3d(
                    x=x_all,
                    y=y_all,
                    z=z_all,
                    i=i_all,
                    j=j_all,
                    k=k_all,
                    opacity=0.7,
                    color='lightblue',
                    flatshading=True,
                    lighting=dict(ambient=0.5, diffuse=0.8, specular=0.2),
                    showlegend=False
                ))
    
    frames.append(go.Frame(data=traces, name=str(year)))
    print(f"Created frame for {year} with {len(traces)} buildings")

# Create figure with first frame
fig = go.Figure(
    data=frames[0].data if frames else [],
    frames=frames
)

# Add slider and play button
fig.update_layout(
    title=dict(
        text='New Cairo Building Height Evolution (2016-2023)',
        x=0.5,
        xanchor='center'
    ),
    scene=dict(
        xaxis=dict(title='', showticklabels=False, showgrid=False),
        yaxis=dict(title='', showticklabels=False, showgrid=False),
        zaxis=dict(title='Height (m)', showgrid=True),
        aspectmode='data',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2),
            center=dict(x=0, y=0, z=0)
        )
    ),
    updatemenus=[
        dict(
            type='buttons',
            showactive=False,
            buttons=[
                dict(
                    label='Play',
                    method='animate',
                    args=[None, {
                        'frame': {'duration': 800, 'redraw': True},
                        'fromcurrent': True,
                        'mode': 'immediate'
                    }]
                ),
                dict(
                    label='Pause',
                    method='animate',
                    args=[[None], {
                        'frame': {'duration': 0, 'redraw': False},
                        'mode': 'immediate'
                    }]
                )
            ],
            x=0.1,
            y=0,
            xanchor='left',
            yanchor='bottom'
        )
    ],
    sliders=[{
        'active': 0,
        'steps': [
            {
                'args': [[f.name], {
                    'frame': {'duration': 0, 'redraw': True},
                    'mode': 'immediate'
                }],
                'label': f.name,
                'method': 'animate'
            }
            for f in frames
        ],
        'x': 0.1,
        'len': 0.9,
        'xanchor': 'left',
        'y': 0,
        'yanchor': 'top'
    }]
)

fig.show()

In [ ]:
# Optional: Save as HTML
fig.write_html('new_cairo_timelapse.html')
print("\nTimelapse saved as 'new_cairo_timelapse.html'")

### Trial 2

In [ ]:
import geopandas as gpd
import plotly.graph_objects as go
import numpy as np
from pathlib import Path
import re

# Load all GeoJSON files from the Buildings folder
buildings_folder = Path('Buildings')
building_files = sorted(buildings_folder.glob('NewCairo_Building_Height_*.geojson'))

print(f"Found {len(building_files)} files")

# Load all years of data
all_buildings = []
for file in building_files:
    # Extract year from filename
    year_match = re.search(r'(\d{4})', file.name)
    if year_match:
        year = int(year_match.group(1))
        
        gdf = gpd.read_file(file)
        gdf['year'] = year
        all_buildings.append(gdf)
        print(f"Loaded {len(gdf)} buildings from {year}")

# Combine all data
import pandas as pd
buildings_all = pd.concat(all_buildings, ignore_index=True)

# Project to a local coordinate system for better visualization
buildings_projected = buildings_all.to_crs('EPSG:3857')  # Web Mercator

# Get the center for visualization (use projected data to avoid warning)
center_point = buildings_projected.geometry.centroid
center_lon = buildings_all.geometry.centroid.x.mean()
center_lat = buildings_all.geometry.centroid.y.mean()

print(f"\nTotal buildings across all years: {len(buildings_all)}")
print(f"Center coordinates: {center_lat:.4f}, {center_lon:.4f}")
print(f"Years: {sorted(buildings_all['year'].unique())}")

# Create animation frames
years = sorted(buildings_all['year'].unique())
frames = []

# Calculate bounds for basemap
minx, miny, maxx, maxy = buildings_projected.total_bounds

# Create a ground plane (basemap)
ground_size = max(maxx - minx, maxy - miny) * 1.2
ground_x = [minx - ground_size*0.1, maxx + ground_size*0.1, maxx + ground_size*0.1, minx - ground_size*0.1]
ground_y = [miny - ground_size*0.1, miny - ground_size*0.1, maxy + ground_size*0.1, maxy + ground_size*0.1]

# Basemap trace (will be in every frame)
basemap_trace = go.Mesh3d(
    x=ground_x + ground_x,
    y=ground_y + ground_y,
    z=[0, 0, 0, 0, 0, 0, 0, 0],
    i=[0, 0],
    j=[1, 2],
    k=[2, 3],
    color='#2d5016',  # Dark green for ground
    opacity=0.8,
    showlegend=False,
    hoverinfo='skip'
)

for year in years:
    buildings_year = buildings_projected[buildings_projected['year'] == year]
    
    # Create traces for this year (start with basemap)
    traces = [basemap_trace]
    
    for idx, building in buildings_year.iterrows():
        if building.geometry.geom_type == 'Polygon':
            coords = list(building.geometry.exterior.coords)
            x = [c[0] for c in coords]
            y = [c[1] for c in coords]
            height = building.get('Heightmean', 0) * 3  # Exaggerate height 3x
            
            if height > 0:  # Only show buildings with height
                # Create building as a Mesh3d
                n = len(x)
                
                # Bottom vertices
                x_all = x + x
                y_all = y + y
                z_all = [0] * n + [height] * n
                
                # Create faces (triangles)
                # Bottom face
                i_bottom = [0] * (n - 2)
                j_bottom = list(range(1, n - 1))
                k_bottom = list(range(2, n))
                
                # Top face
                i_top = [n] * (n - 2)
                j_top = [n + i for i in range(2, n)]
                k_top = [n + i for i in range(1, n - 1)]
                
                # Side faces
                i_side = []
                j_side = []
                k_side = []
                for idx in range(n - 1):
                    # Two triangles per side
                    i_side.extend([idx, idx])
                    j_side.extend([idx + 1, n + idx + 1])
                    k_side.extend([n + idx, idx + 1])
                
                i_all = i_bottom + i_top + i_side
                j_all = j_bottom + j_top + j_side
                k_all = k_bottom + k_top + k_side
                
                traces.append(go.Mesh3d(
                    x=x_all,
                    y=y_all,
                    z=z_all,
                    i=i_all,
                    j=j_all,
                    k=k_all,
                    opacity=0.85,
                    color='#87CEEB',  # Sky blue for buildings
                    flatshading=True,
                    lighting=dict(ambient=0.6, diffuse=0.9, specular=0.3),
                    showlegend=False
                ))
    
    frames.append(go.Frame(data=traces, name=str(year)))
    print(f"Created frame for {year} with {len(traces)} buildings")

# Create figure with first frame
fig = go.Figure(
    data=frames[0].data if frames else [],
    frames=frames
)

# Add slider and play button
fig.update_layout(
    title=dict(
        text='New Cairo Building Height Evolution (2016-2023)',
        x=0.5,
        xanchor='center'
    ),
    scene=dict(
        xaxis=dict(title='', showticklabels=False, showgrid=False, showbackground=False),
        yaxis=dict(title='', showticklabels=False, showgrid=False, showbackground=False),
        zaxis=dict(title='Height (m, exaggerated 3x)', showgrid=True, gridcolor='lightgray'),
        aspectmode='data',
        bgcolor='#87CEEB',  # Sky blue background
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2),
            center=dict(x=0, y=0, z=0)
        )
    ),
    updatemenus=[
        dict(
            type='buttons',
            showactive=False,
            buttons=[
                dict(
                    label='Play',
                    method='animate',
                    args=[None, {
                        'frame': {'duration': 800, 'redraw': True},
                        'fromcurrent': True,
                        'mode': 'immediate'
                    }]
                ),
                dict(
                    label='Pause',
                    method='animate',
                    args=[[None], {
                        'frame': {'duration': 0, 'redraw': False},
                        'mode': 'immediate'
                    }]
                )
            ],
            x=0.1,
            y=0,
            xanchor='left',
            yanchor='bottom'
        )
    ],
    sliders=[{
        'active': 0,
        'steps': [
            {
                'args': [[f.name], {
                    'frame': {'duration': 0, 'redraw': True},
                    'mode': 'immediate'
                }],
                'label': f.name,
                'method': 'animate'
            }
            for f in frames
        ],
        'x': 0.1,
        'len': 0.9,
        'xanchor': 'left',
        'y': 0,
        'yanchor': 'top'
    }]
)

fig.show()

# Optional: Save as HTML
fig.write_html('new_cairo_timelapse.html')
print("\nTimelapse saved as 'new_cairo_timelapse.html'")

# Print statistics
print("\n=== Building Statistics by Year ===")
for year in years:
    year_data = buildings_all[buildings_all['year'] == year]
    print(f"{year}: {len(year_data)} buildings, "
          f"avg height: {year_data['Heightmean'].mean():.1f}m, "
          f"max height: {year_data['Heightmean'].max():.1f}m")

### Trial 3

In [ ]:
import geopandas as gpd
import plotly.graph_objects as go
import numpy as np
from pathlib import Path
import re
import pandas as pd
from PIL import Image
import requests
from io import BytesIO

# Load all GeoJSON files from the Buildings folder
buildings_folder = Path('Buildings')
building_files = sorted(buildings_folder.glob('NewCairo_Building_Height_*.geojson'))

print(f"Found {len(building_files)} files")

# Load all years of data
all_buildings = []
for file in building_files:
    # Extract year from filename
    year_match = re.search(r'(\d{4})', file.name)
    if year_match:
        year = int(year_match.group(1))
        
        gdf = gpd.read_file(file)
        gdf['year'] = year
        all_buildings.append(gdf)
        print(f"Loaded {len(gdf)} buildings from {year}")

# Combine all data
buildings_all = pd.concat(all_buildings, ignore_index=True)

# Project to a local coordinate system for better visualization
buildings_projected = buildings_all.to_crs('EPSG:3857')  # Web Mercator

# Get the center for visualization (use projected data to avoid warning)
center_point = buildings_projected.geometry.centroid
center_lon = buildings_all.geometry.centroid.x.mean()
center_lat = buildings_all.geometry.centroid.y.mean()

print(f"\nTotal buildings across all years: {len(buildings_all)}")
print(f"Center coordinates: {center_lat:.4f}, {center_lon:.4f}")
print(f"Years: {sorted(buildings_all['year'].unique())}")

# Calculate bounds for basemap
minx, miny, maxx, maxy = buildings_projected.total_bounds
width = maxx - minx
height = maxy - miny

# Fetch satellite imagery for the area
print("\nFetching satellite imagery...")
try:
    # Calculate zoom level and tile coordinates for the area
    zoom = 15  # Adjust for resolution
    
    # Use ArcGIS World Imagery (free, no token needed)
    # Create a simple textured ground plane
    # For actual satellite imagery, you'd need to implement proper tile fetching
    # This creates a darker ground plane with grid to simulate terrain
    
    # Create ground plane vertices
    ground_res = 50  # Grid resolution
    x_ground = np.linspace(minx - width*0.1, maxx + width*0.1, ground_res)
    y_ground = np.linspace(miny - height*0.1, maxy + height*0.1, ground_res)
    X_ground, Y_ground = np.meshgrid(x_ground, y_ground)
    Z_ground = np.zeros_like(X_ground) - 5  # Slightly below buildings
    
    # Create a simple color pattern (darker green-brown terrain color)
    # You can replace this with actual satellite imagery if needed
    colorscale = [[0, '#1a2b1a'], [0.5, '#2d3d2d'], [1, '#3a4a3a']]
    
    print("Created ground plane")
    
except Exception as e:
    print(f"Using simple ground plane: {e}")
    ground_res = 10
    x_ground = np.linspace(minx - width*0.1, maxx + width*0.1, ground_res)
    y_ground = np.linspace(miny - height*0.1, maxy + height*0.1, ground_res)
    X_ground, Y_ground = np.meshgrid(x_ground, y_ground)
    Z_ground = np.zeros_like(X_ground) - 5
    colorscale = [[0, '#1a2b1a'], [1, '#2d3d2d']]

# Create animation frames
years = sorted(buildings_all['year'].unique())
frames = []

# Basemap trace (terrain-style ground plane)
basemap_trace = go.Surface(
    x=X_ground,
    y=Y_ground,
    z=Z_ground,
    colorscale=colorscale,
    showscale=False,
    lighting=dict(ambient=0.9, diffuse=0.5, specular=0.1),
    hoverinfo='skip',
    showlegend=False
)

for year in years:
    buildings_year = buildings_projected[buildings_projected['year'] == year]
    
    # Create traces for this year (start with basemap)
    traces = [basemap_trace]
    
    for idx, building in buildings_year.iterrows():
        if building.geometry.geom_type == 'Polygon':
            coords = list(building.geometry.exterior.coords)
            x = [c[0] for c in coords]
            y = [c[1] for c in coords]
            height = building.get('Heightmean', 0) * 3  # Exaggerate height 3x
            
            if height > 0:  # Only show buildings with height
                # Create building as a Mesh3d
                n = len(x)
                
                # Bottom vertices
                x_all = x + x
                y_all = y + y
                z_all = [0] * n + [height] * n
                
                # Create faces (triangles)
                # Bottom face
                i_bottom = [0] * (n - 2)
                j_bottom = list(range(1, n - 1))
                k_bottom = list(range(2, n))
                
                # Top face
                i_top = [n] * (n - 2)
                j_top = [n + i for i in range(2, n)]
                k_top = [n + i for i in range(1, n - 1)]
                
                # Side faces
                i_side = []
                j_side = []
                k_side = []
                for idx in range(n - 1):
                    # Two triangles per side
                    i_side.extend([idx, idx])
                    j_side.extend([idx + 1, n + idx + 1])
                    k_side.extend([n + idx, idx + 1])
                
                i_all = i_bottom + i_top + i_side
                j_all = j_bottom + j_top + j_side
                k_all = k_bottom + k_top + k_side
                
                traces.append(go.Mesh3d(
                    x=x_all,
                    y=y_all,
                    z=z_all,
                    i=i_all,
                    j=j_all,
                    k=k_all,
                    opacity=0.9,
                    color='#1e3a5f',  # Dark blue for buildings
                    flatshading=True,
                    lighting=dict(ambient=0.4, diffuse=0.8, specular=0.5),
                    showlegend=False
                ))
    
    frames.append(go.Frame(data=traces, name=str(year)))
    print(f"Created frame for {year} with {len(traces)-1} buildings")

# Create figure with first frame
fig = go.Figure(
    data=frames[0].data if frames else [],
    frames=frames
)

# Add slider and play button
fig.update_layout(
    title=dict(
        text='New Cairo Building Height Evolution (2016-2023)',
        x=0.5,
        xanchor='center',
        font=dict(size=20, color='white')
    ),
    paper_bgcolor='#0a0a0a',
    scene=dict(
        xaxis=dict(title='', showticklabels=False, showgrid=False, showbackground=False),
        yaxis=dict(title='', showticklabels=False, showgrid=False, showbackground=False),
        zaxis=dict(
            title='Height (m, exaggerated 3x)', 
            showgrid=True, 
            gridcolor='rgba(255,255,255,0.1)',
            backgroundcolor='#0a0a0a'
        ),
        aspectmode='data',
        bgcolor='#1a1a2e',  # Dark background
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2),
            center=dict(x=0, y=0, z=0)
        )
    ),
    updatemenus=[
        dict(
            type='buttons',
            showactive=False,
            buttons=[
                dict(
                    label='▶ Play',
                    method='animate',
                    args=[None, {
                        'frame': {'duration': 800, 'redraw': True},
                        'fromcurrent': True,
                        'mode': 'immediate'
                    }]
                ),
                dict(
                    label='⏸ Pause',
                    method='animate',
                    args=[[None], {
                        'frame': {'duration': 0, 'redraw': False},
                        'mode': 'immediate'
                    }]
                )
            ],
            x=0.1,
            y=0,
            xanchor='left',
            yanchor='bottom',
            bgcolor='rgba(255,255,255,0.1)',
            font=dict(color='white')
        )
    ],
    sliders=[{
        'active': 0,
        'steps': [
            {
                'args': [[f.name], {
                    'frame': {'duration': 0, 'redraw': True},
                    'mode': 'immediate'
                }],
                'label': f.name,
                'method': 'animate'
            }
            for f in frames
        ],
        'x': 0.1,
        'len': 0.9,
        'xanchor': 'left',
        'y': 0,
        'yanchor': 'top',
        'bgcolor': 'rgba(255,255,255,0.1)',
        'font': dict(color='white')
    }]
)

fig.show()

# Optional: Save as HTML
fig.write_html('new_cairo_timelapse.html')
print("\nTimelapse saved as 'new_cairo_timelapse.html'")

# Print statistics
print("\n=== Building Statistics by Year ===")
for year in years:
    year_data = buildings_all[buildings_all['year'] == year]
    print(f"{year}: {len(year_data)} buildings, "
          f"avg height: {year_data['Heightmean'].mean():.1f}m, "
          f"max height: {year_data['Heightmean'].max():.1f}m")

print("\n💡 Tip: For actual satellite imagery basemap, you can:")
print("   1. Use contextily to fetch tiles and create a texture")
print("   2. Or use Pydeck which has native satellite support")